In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/cafa-6-protein-function-prediction/sample_submission.tsv
/kaggle/input/cafa-6-protein-function-prediction/IA.tsv
/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset.fasta
/kaggle/input/cafa-6-protein-function-prediction/Test/testsuperset-taxon-list.tsv
/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv
/kaggle/input/cafa-6-protein-function-prediction/Train/train_sequences.fasta
/kaggle/input/cafa-6-protein-function-prediction/Train/train_taxonomy.tsv
/kaggle/input/cafa-6-protein-function-prediction/Train/go-basic.obo
/kaggle/input/cafa-5-protein-function-with-tensorflow/submission.tsv
/kaggle/input/cafa-5-protein-function-with-tensorflow/__results__.html
/kaggle/input/cafa-5-protein-function-with-tensorflow/__resultx__.html
/kaggle/input/cafa-5-protein-function-with-tensorflow/__notebook__.ipynb
/kaggle/input/cafa-5-protein-function-with-tensorflow/__output__.json
/kaggle/input/cafa-5-protein-function-with-tensorflow/custom.css
/kaggle/i

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
import os

# --- SETTINGS ---
NUM_LABELS = 1500
BATCH_SIZE = 256
EPOCHS = 12
N_FOLDS = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- LOAD T5 EMBEDDINGS ONLY ---
print("📦 Loading T5 embeddings...")
t5_path = '/kaggle/input/t5embeds'
X_all = np.load(os.path.join(t5_path, 'train_embeds.npy'))
X_test = np.load(os.path.join(t5_path, 'test_embeds.npy'))
train_ids = np.load(os.path.join(t5_path, 'train_ids.npy'))
test_ids = np.load(os.path.join(t5_path, 'test_ids.npy'))

# Labels
train_terms = pd.read_csv("/kaggle/input/cafa-6-protein-function-prediction/Train/train_terms.tsv", sep="\t")
all_go_terms = train_terms['term'].value_counts().head(NUM_LABELS).index.tolist()

pivot = train_terms[train_terms['term'].isin(all_go_terms)].pivot(index='EntryID', columns='term', values='term').notna().astype(np.int8)
pivot = pivot.reindex(index=train_ids, columns=all_go_terms, fill_value=0)
y_all = pivot.values.astype(np.float32)

INPUT_DIM = X_all.shape[1]

# --- MODEL ARCHITECTURE ---
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
    def forward(self, x): return x + self.block(x)

class Top5Model(nn.Module):
    def __init__(self, input_dim, num_labels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            ResidualBlock(1024),
            nn.Linear(1024, 2048),
            nn.BatchNorm1d(2048),
            nn.ReLU(),
            ResidualBlock(2048),
            nn.Linear(2048, num_labels),
            nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

# --- 5-FOLD CROSS-VALIDATION ---
kfold = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
all_test_preds = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_all)):
    print(f"\n🌀 Training Fold {fold+1}/{N_FOLDS}...")
    
    X_train, y_train = X_all[train_idx], y_all[train_idx]
    train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train).float()),
                              batch_size=BATCH_SIZE, shuffle=True)
    
    model = Top5Model(INPUT_DIM, NUM_LABELS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCELoss()
    
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        if (epoch+1) % 4 == 0:
            print(f"  Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.4f}")
    
    # Fold inference
    model.eval()
    fold_preds = []
    with torch.no_grad():
        test_tensor = torch.from_numpy(X_test).float().to(device)
        for i in range(0, len(test_tensor), BATCH_SIZE):
            batch = test_tensor[i:i+BATCH_SIZE]
            fold_preds.append(model(batch).cpu().numpy())
    
    all_test_preds.append(np.concatenate(fold_preds))
    print(f"✅ Fold {fold+1} Complete.")

# --- ENSEMBLE & WEIGHTED SUBMISSION ---
print("\n🤖 Averaging predictions across folds...")
final_preds = np.mean(all_test_preds, axis=0)

go_freq = train_terms['term'].value_counts(normalize=True).to_dict()
top_go_terms = list(pd.Series(go_freq).sort_values(ascending=False).head(100).index)
boost_factor = 1.05

top_k_main = 30
top_k_extra = 50
main_threshold = 0.05
extra_threshold = 0.1
submission_file = "submission.tsv"

with open(submission_file, "w") as f:
    for i in range(len(final_preds)):
        prot_id = test_ids[i]
        top_indices = np.argsort(final_preds[i])[::-1][:top_k_extra]
        for rank, idx in enumerate(top_indices):
            score = final_preds[i][idx]
            term = all_go_terms[idx]
            # Weighted boost
            if term in top_go_terms:
                score *= boost_factor
                score = min(score, 1.0)
            # Threshold logic
            if rank < top_k_main and score > main_threshold:
                f.write(f"{prot_id}\t{term}\t{score:.3f}\n")
            elif rank >= top_k_main and score > extra_threshold:
                f.write(f"{prot_id}\t{term}\t{score:.3f}\n")

print(f"🚀 Submission ready: '{submission_file}' (Top-5 target, T5-only)")


📦 Loading T5 embeddings...

🌀 Training Fold 1/5...
  Epoch 4 Loss: 0.0069
  Epoch 8 Loss: 0.0060
  Epoch 12 Loss: 0.0049
✅ Fold 1 Complete.

🌀 Training Fold 2/5...
  Epoch 4 Loss: 0.0068
  Epoch 8 Loss: 0.0059
  Epoch 12 Loss: 0.0049
✅ Fold 2 Complete.

🌀 Training Fold 3/5...
  Epoch 4 Loss: 0.0069
  Epoch 8 Loss: 0.0059
  Epoch 12 Loss: 0.0049
✅ Fold 3 Complete.

🌀 Training Fold 4/5...
  Epoch 4 Loss: 0.0069
  Epoch 8 Loss: 0.0060
  Epoch 12 Loss: 0.0049
✅ Fold 4 Complete.

🌀 Training Fold 5/5...
  Epoch 4 Loss: 0.0069
  Epoch 8 Loss: 0.0060
  Epoch 12 Loss: 0.0049
✅ Fold 5 Complete.

🤖 Averaging predictions across folds...
🚀 Submission ready: 'submission.tsv' (Top-5 target, T5-only)
